In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

Given the images provided by Mr. Rivero in a Google Drive, create a Machine Learning model that can predict what character is written in the 
cursive Englis
h alphabet for a given letter. The program must be written in Python.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

I downloaded the Cursive folder given by Rivero into a zip file. I then directly uploaded th
e contents of each student (for example, S1) subfolder
directly to Jupyter Notebook in the data folder, unzipping the files with ease using my MacBook's file selection tools.

# 3. Explore the Data
Gain insights into
the data you have from step 2, making sure to identify any bias


Insights:
When I unzipped the Cursive file, I noticed many duplicates, which I removed
as it can set an unequal distribution when traning the model.
Looking at the data, I notice there are many different file types such as.png, .PNG, .jpeg, .HEIC, .heic, .jpg, and .JPG, with some not even being a valid file. The fact that so many are not valid files can lead to a smaller dataset which can reduce the accuracy in the long run. Moreover, I notice that the images are very different; some show a large portion of the table, some are already cropped to the middle, and some show a bit of table, and some have lots of noise. Some of the handwriting is illegible, which can skew the learning process later on.

Bias:
I am biased in that I believe some of the handwriting renders some data unusuable; I feel it is completely illegible. However, this can either be important information as the model can learn how to recognize characters despite poor handwriting, or be a detriment to the model 
training process.


# 4.Prepare the Data


Apply any data transformations and explain what and why


In [25]:
import cv2
import numpy as np
import os
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()
data = "/home/jupyter-1002215/Project_5/data"
output = "/home/jupyter-1002215/Project_5/output"
output_path= "/home/jupyter-1002215/Project_5/output_processed"

os.makedirs(output_path, exist_ok=True)

files = [f for f in os.listdir(data)]

# Create output directory if it does not exist
if not os.path.exists(os.path.join(data, 'output')):
    os.makedirs(os.path.join(data, 'output'))

# Convert each file to JPEG
for filename in files:
    try:
        image = Image.open(os.path.join(data, filename))
        image.convert('RGB').save(os.path.join(data, 'output', os.path.splitext(filename)[0] + '.jpg'))
    except Exception as e:
        print(e)

# Standard dimesnions given by Rivero
target_size = (224, 224)
count=0

for filename in os.listdir(output):
    try:
        count+=1;
        image = os.path.join(output, filename)
        base_name, _ = os.path.splitext(filename)
        out = os.path.join(output_path, f"{base_name}.jpg")
        img = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        
        blurred = cv2.GaussianBlur(img, (5, 5), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) # Find contours to detect letter

        largest_contour = max(contours, key=cv2.contourArea) # Largest contour is the letter
        x, y, w, h = cv2.boundingRect(largest_contour)
        cropped = binary[y:y+h, x:x+w]
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_AREA)
    
        cv2.imwrite(out, resized)
    except Exception as e:
        print(e)
print(count)


Decoder plugin generated an error: Unexpected end of file
[Errno 21] Is a directory: '/home/jupyter-1002215/Project_5/data/.ipynb_checkpoints'
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
Decoder plugin generated an error: Unexpected end of file
OpenCV(4.12.0) /io/opencv/modules/imgproc/src/smooth.dispatch.cpp:618: error: (-215:Assertion failed) !_src.empty() in function 'GaussianBlur'

462


For each file in my data folder, which contains images of cursive characters from each student in the class, I first
    converted them to a jpeg file then
    grayscale to simplify processing. I then applied Gaussian blur to reduce noise. I used Otsu
    binarization to convert each pixel in the image to binary, either 0 or 1. The reason I did this was because
    when I apply the neural netwrok later on, each pixel of each image will be an attribute and each can have a value
    of 0 or 1, allowing the computer to easily process each state and learn more efficiently. I then took all the contours
    and since the letter is generally the largest contour, I extracted the dimensions of the largest contour and resized my
    image to the dimensions recommended by mr. Rivero. All of the processed images were saved to the output_processed folder.

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [24]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import pickle

folder = 'output_processed'
files = [f for f in os.listdir(folder) if f.endswith('.jpg')]

data = []
for f in files:
    label = f[0]  # First character as label
    path = os.path.join(folder, f)
    data.append([path, label])

df = pd.DataFrame(data, columns=['Filepath', 'Label'])
df.to_csv('cursive.csv', index=False)

datagen =ImageDataGenerator(
        rescale=1./255,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        validation_split=0.55
        )

train_gen =datagen.flow_from_dataframe(
        df,
        x_col='Filepath',
        y_col='Label',
        target_size=(224,224),
        color_mode='grayscale',
        class_mode='categorical',
        batch_size=16,
        shuffle=True,
        subset='training'
        )

val_gen =datagen.flow_from_dataframe(
        df,
        x_col='Filepath',
        y_col='Label',
        target_size=(224,224),
        color_mode='grayscale',
        class_mode='categorical',
        batch_size=16,
        shuffle=False,
        subset='validation'
        )

num_classes = len(train_gen.class_indices)

model =models.Sequential([
        layers.Conv2D(16, (3,3), activation='relu', input_shape=(224,224,1)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
        ])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)

train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"Training accuracy: {train_acc*100:.2f}%")
print(f"Validation accuracy: {val_acc*100:.2f}%")

with open('project5_model.pkl','wb') as f:
    pickle.dump(model, f)


Found 208 validated image filenames belonging to 52 classes.
Found 253 validated image filenames belonging to 52 classes.
Model: "sequential_16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_56 (Conv2D)          (None, 222, 222, 16)      160       
                                                                 
 max_pooling2d_53 (MaxPooli  (None, 111, 111, 16)      0         
 ng2D)                                                           
                                                                 
 conv2d_57 (Conv2D)          (None, 109, 109, 32)      4640      
                                                                 
 max_pooling2d_54 (MaxPooli  (None, 54, 54, 32)        0         
 ng2D)                                                           
                                                                 
 conv2d_58 (Conv2D)          (None, 52, 52, 64)        18496   

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


First, I labeled each file by the first letter of the file name since I manually labeled the unlabeled files to start with
their respective letter, taking lower and uppercase into account. Then, I used ImageDataGenerator() for even more preparation for
training, splitting the training and validation for 20%. I generated a batch of augmented images from the dataframe for both the training
set and validation set. I then specified a stack of layers, first a convolution layer with 16 filters, 3x3 kernel, and ReLU activation, then downsampling feature maps by a factor of 2, then increasing filter depth to get more features, and flattening the 2D maps into 1D vector. I connected the layers and randomly set 30% of neurons to 0 to curb overfitting. My output layer used softmax activation for multi-class classification. My accuracy was initially very low, about 10%, but by experimenting with the number of filters and test_train_split, I managed to get a training accuracy of 24.04% and a validation accuracy of 6.32%. These values are all extremely low
due to my limited dataset. I tried to use the dataset from the Cursive
Character Challenge with 55,000+ characters 
but Jupyter could not support its sheer size and I had several issues with labeling; I ultimately ran out of time and I take full accountability.

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


In [ ]:
Summary of Findings:
My product uses Machine Learning to detect and predict cursive letters,
be it uppercase or lowercase. 
Detail Approach Taken:
Using a dataset provided by Mr. Rivero, I labeled and preprocessed files of
cursive characters for my machine learning model. I ultimately generated more 
training and testing data from this dataset and applied a Convolutional Neural
Network, specializing in pattern recognition.


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [41]:
import string

with open('project5_model.pkl', 'rb') as f:
    loaded_model=pickle.load(f)

classes = list(string.ascii_uppercase) + list(string.ascii_lowercase)
target_size = (224, 224)  # define target size

def inference(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    img = img[y:y+h, x:x+w]

    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    img = img / 255.0
    img = np.expand_dims(img, axis=(0, -1))  # shape: (1, 224, 224, 1)

    prediction = loaded_model.predict(img)
    predicted_class = np.argmax(prediction, axis=1)[0]
    predicted_letter = classes[predicted_class]
    print("Predicted letter:", predicted_letter)  # <-- use predicted_letter here

inference("test.jpg")


1/1 [==============================] - 0s 88ms/step
Predicted letter: l
